# Korean sLLM 학습 (Gemma-계열 + 16-토큰 MTP)

GitHub 리포를 clone 하고 tar.xz 데이터를 풀어 학습한다. **RTX PRO 6000 (96GB) 런타임 기준** — T4 등 16GB GPU 는 이 설정(mtp 16 + seq 2048)의 로짓 메모리를 감당하지 못한다.

**설정 근거** ([docs/seq_len_review.md](docs/seq_len_review.md), [docs/model_config_review.md](docs/model_config_review.md), 수치는 vocab 32,768·coverage 0.9999 최종 토크나이저 실측):

- `--seq-len 2048`: 전체 train 301,669 샘플의 99.76% 가 2048 tokens 이내(p99 1,246). 단순 패킹 시 윈도우 경계 잘림 ≈12%.
- `--max-sample-len 2048`(기본 = seq_len): 2048 초과 721 샘플(0.24%)은 캐시 단계에서 제외. 추론 시 처리 불가한 길이이고, 패킹하면 질문 없는 답변 조각만 남기기 때문.
- 토큰 예산: train 전체 76.86M, 필터 후 ≈74.86M tokens. `batch 8 × grad_accum 4 × 2048 = 65,536 tokens/step` → **1 epoch ≈ 1,142 steps**. 정확한 값은 학습 시작 배너의 "스케줄" 줄에 찍힌다. `--epochs 4`(≈4,570 steps) 로 두고 val main_loss 를 보며 조정한다.
- 메모리: mtp 16 의 CE 로짓(fp32)이 지배해 batch 8 기준 ≈51 GiB 추정. **grad-checkpointing 불필요**, batch 16 은 96GB 초과라 8×4 유지. 첫 eval 로그의 `mem` 값으로 실측 확인.
- 답변 길이 p90 392 / p99 839 tokens → 생성 데모는 `max_new_tokens=512`.
- 체크포인트는 `best.pt`(val main_loss 최저) + `last.pt`(최신) 두 개만 유지·덮어쓰기된다. 최종 모델은 `best.pt`, 재개는 `last.pt`.
- Drive 에 저장한 체크포인트는 마지막 셀 `drive.flush_and_unmount()` 까지 실행해야 확실히 반영된다.

In [ ]:
# 1) 리포 clone (본인 리포 URL 로 수정)
REPO_URL = "https://github.com/MinsuChae/korean_sllm.git"
!git clone {REPO_URL} korean_sllm
%cd korean_sllm
!pip install -q -r requirements.txt

In [ ]:
# 2) Google Drive 마운트 - 체크포인트 보존용
#    주의: Drive 에 쓴 파일은 마지막 셀의 drive.flush_and_unmount() 를 실행해야 확실히 반영된다.
#    (런타임이 끊기면 flush 안 된 쓰기가 유실될 수 있으니 학습이 길면 중간에도 drive.flush_and_unmount() 후 재마운트)
from google.colab import drive
drive.mount('/content/drive')
CKPT_DIR = '/content/drive/MyDrive/korean_sllm_ckpt'

In [ ]:
# 3) 데이터 압축 해제 (train.py 가 자동으로 풀지만 미리 풀어 확인)
!tar -xJf train.tar.xz && tar -xJf val.tar.xz
!wc -l train.jsonl val.jsonl

In [ ]:
# 4) 학습 (RTX PRO 6000 96GB 기준: batch 8 × accum 4, grad-checkpointing 없음, 예상 ≈51 GiB)
#    --epochs 4: 데이터셋 윈도우 수에서 max_steps 를 자동 환산 (시작 배너의 "스케줄" 줄 확인).
#    체크포인트는 last.pt(--save-every 주기 최신)와 best.pt(val main_loss 최저) 두 개만 유지된다.
#    val main_loss 가 계속 내려가면 --resume + --epochs 상향으로 연장. 상승 전환해도 best.pt 가 최고 성능 시점을 보존한다.
#    --max-sample-len 은 기본값(= seq_len 2048)으로 두면 2048 초과 샘플이 캐시에서 제외된다.
!python train.py \
  --seq-len 2048 \
  --batch-size 8 --grad-accum 4 \
  --epochs 4 --warmup-steps 300 --lr 3e-4 \
  --eval-every 250 --save-every 500 \
  --ckpt-dir {CKPT_DIR}

# 재개: --resume {CKPT_DIR}/last.pt 추가 (연장 시 --epochs 도 함께 올릴 것. best.pt 재개는 과거 시점으로 되돌아가니 lr 조정 등 의도된 경우만)

In [ ]:
# 5) 생성 데모 - val main_loss 최저 시점인 best.pt 로드. 답변 길이 p90 392 / p99 839 tokens (32k 실측) 이므로 512 이상으로 (256 은 중앙값 134 만 겨우 넘김)
import torch
from data import load_tokenizer, encode_sample
from model import KoreanSLLM, ModelConfig

ckpt = torch.load(f'{CKPT_DIR}/best.pt', map_location='cuda', weights_only=True)
model = KoreanSLLM(ModelConfig(**ckpt['config'])).cuda()
model.load_state_dict(ckpt['model'])
sp = load_tokenizer()

prompt = '감기에 걸렸을 때 어떻게 해야 하나요?'
ids = encode_sample(sp, prompt, '')[0][:-2]
out = model.generate(torch.tensor([ids], device='cuda'), max_new_tokens=512, temperature=0.7)
print(sp.decode(out[0].tolist()))

In [ ]:
# 5b) self-speculative decoding 데모 (batch=1) - MTP 헤드로 draft 를 뽑아 트렁크 1회 forward 로 병렬 검증
#     greedy(temperature=0)는 generate 와 출력 동일이 보장되고, sampling 은 rejection 으로 동일 분포 보존
#     (단, RNG 소비가 달라 아래 두 출력의 내용 자체는 다를 수 있음). draft_k 는 offset별 수용률을 보고 조정.
import time

def timed(fn):
    torch.cuda.synchronize(); t0 = time.perf_counter()
    out = fn()
    torch.cuda.synchronize(); return out, time.perf_counter() - t0

inp = torch.tensor([ids], device='cuda')
base, t_base = timed(lambda: model.generate(inp, max_new_tokens=512, temperature=0.7))
(spec, stats), t_spec = timed(lambda: model.generate_speculative(
    inp, max_new_tokens=512, temperature=0.7, draft_k=8, return_stats=True))

n_base = base.shape[1] - inp.shape[1]
n_spec = spec.shape[1] - inp.shape[1]
acc, prop = sum(stats['accepted']), sum(stats['proposed'])
print(f'generate            : {n_base:4d} tokens, {t_base:.2f}s ({n_base / t_base:.1f} tok/s)')
print(f'generate_speculative: {n_spec:4d} tokens, {t_spec:.2f}s ({n_spec / t_spec:.1f} tok/s)')
print(f'draft 수용률 {acc}/{prop} ({acc / max(prop, 1):.0%}), offset별:',
      ' '.join(f'{k+1}:{a}/{p}' for k, (a, p) in enumerate(zip(stats['accepted'], stats['proposed'])) if p))
print()
print(sp.decode(spec[0].tolist()))

In [ ]:
# 6) Drive 에 쓴 체크포인트를 확실히 저장 - 버퍼 flush 후 언마운트 (이후 Drive 경로 접근 불가, 필요하면 2번 셀로 재마운트)
import os
print('저장된 체크포인트:', sorted(f for f in os.listdir(CKPT_DIR) if f.endswith('.pt')))
drive.flush_and_unmount()
print('Drive flush + unmount 완료')